## __Get Macro data__

In [11]:
#df = pd.read_parquet('./data/raw/macro_raw.parquet')
#print(df['datetime_utc'].nunique())
#print(df['datetime_utc'].value_counts().head())


In [16]:
"""
collect_macro_data.py

Collecte macro "end-to-end":
 - RSS feeds (ECB, IMF, Reuters...)
 - Scraping officiels (FOMC statements/minutes, ECB press releases)
 - Twitter/X scraping via tweepy (comptes officiels) - requires TWITTER_BEARER_TOKEN env var
 - Language detection, optional translation using googletrans
 - FinBERT tone scoring (hawkish/dovish proxy)
 - Sentence-transformers embeddings
 - Store: raw parquet and daily aggregated parquet signals

Sorties:
 - /data/raw/macro_raw.parquet
 - /data/daily/macro_daily_signals.parquet
"""

import os
import json
from datetime import datetime, timezone, timedelta
import time
import re
import logging
from typing import List, Dict, Any

import feedparser
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from langdetect import detect, DetectorFactory
from newspaper import Article
from tqdm import tqdm

# NLP
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

# Twitter
import tweepy

# Translation
from googletrans import Translator

# Ensure deterministic langdetect
DetectorFactory.seed = 0

# -------------------------
# Configuration
# -------------------------
OUTPUT_RAW = "./data/raw/macro_raw.parquet"
OUTPUT_DAILY = "./data/daily/macro_daily_signals.parquet"
os.makedirs(os.path.dirname(OUTPUT_RAW), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_DAILY), exist_ok=True)

# RSS & scraping sources (extend as needed)
RSS_FEEDS = {
    "ECB_press": "https://www.ecb.europa.eu/rss/press.xml",
    "IMF_news": "https://www.imf.org/external/rss/bottom.xml",  # Corrected to a valid feed
    # Removed Reuters as public RSS is unavailable; consider alternatives like Bloomberg
}

# Official pages to scrape (examples)
SCRAPE_PAGES = {
    "FOMC_press": "https://www.federalreserve.gov/newsevents/pressreleases.htm",
    "FOMC_minutes": "https://www.federalreserve.gov/monetarypolicy/fomc_historical_year.htm",
    "ECB_press": "https://www.ecb.europa.eu/press/releases/html/index.en.html",
}

# Twitter accounts to scrape via tweepy (officials)
TWITTER_ACCOUNTS = [
    "federalreserve", "ecb", "bankofengland", "boj_official_en"  # adapt
]

# Model names
FINBERT_MODEL = "yiyanghkust/finbert-tone"                 # tone classifier (neg/neu/pos)
SENTENCE_EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # cheap and fast embeddings

# Credibility scores for sources (used for weighting)
SOURCE_CREDIBILITY = {
    "ECB_press": 1.0,
    "FOMC_press": 1.0,
    "IMF_news": 0.9,
    "Reuters_economy": 0.95,
    "Twitter": 0.6,
    "other": 0.5
}

# Tokenizer chunk size for long texts
MAX_TOKENS = 500  # conservative chunking for FinBERT tokenizer

# -------------------------
# Helpers: scraping + extraction
# -------------------------
def fetch_rss_entries(feed_url: str) -> List[Dict[str, Any]]:
    feed = feedparser.parse(feed_url)
    entries = []
    for e in feed.entries:
        # Some feeds use published/updated differently
        published = e.get("published") or e.get("updated") or None
        entries.append({
            "title": e.get("title", ""),
            "summary": e.get("summary", "") or e.get("description", ""),
            "link": e.get("link", ""),
            "published": published
        })
    return entries

def fetch_article_text(url: str) -> str:
    # Try newspaper3k first (robust), fallback to requests+bs4
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        if text and len(text) > 50:
            return text
    except Exception:
        pass
    # fallback
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "macro-collector/1.0"})
        r.raise_for_status()
        soup = BeautifulSoup(r.content, "html.parser")
        # remove scripts/styles
        for s in soup(["script", "style"]):
            s.decompose()
        paragraphs = [p.get_text(separator=" ", strip=True) for p in soup.find_all("p")]
        text = "\n\n".join([p for p in paragraphs if len(p) > 20])
        return text
    except Exception:
        return ""

def parse_datetime(s):
    # Try parse many formats
    try:
        dt = pd.to_datetime(s)
        if dt.tzinfo is None:
            dt = dt.tz_localize('UTC')
        return dt
    except Exception:
        return pd.Timestamp(datetime.utcnow(), tz='UTC')

# -------------------------
# NLP: FinBERT scoring + embeddings
# -------------------------
print("Loading NLP models... (this may take a while on first run)")
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    fin_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
    fin_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL).to(device)
    emb_model = SentenceTransformer(SENTENCE_EMB_MODEL, device=device)
except Exception as e:
    logging.error(f"Failed to load NLP models: {e}")
    raise

def chunk_text(text: str, max_tokens: int = MAX_TOKENS) -> List[str]:
    # naive chunk by sentences/paragraphs to keep under token limit
    parts = re.split(r'\n{2,}|\.\s+', text)
    chunks = []
    cur = ""
    for p in parts:
        if len((cur + " " + p).split()) > max_tokens:
            if cur:
                chunks.append(cur.strip())
            cur = p
        else:
            cur = (cur + " " + p).strip()
    if cur:
        chunks.append(cur.strip())
    return [c for c in chunks if len(c) > 20]

def finbert_tone_score(text: str) -> float:
    """
    Return continuous score in [-1, 1] approximating hawkishness:
      - we map FinBERT labels (neg, neu, pos) as (negative->-1, neutral->0, positive->+1)
      - We compute probabilities and use pos_prob - neg_prob as score in [-1,1]
    """
    if not isinstance(text, str) or len(text) < 10:
        return 0.0
    chunks = chunk_text(text)
    scores = []
    for ch in chunks:
        inputs = fin_tokenizer(ch, truncation=True, padding=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = fin_model(**inputs).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy().flatten()
        # model labels order for this checkpoint: [negative, neutral, positive]
        neg, neu, pos = probs[0], probs[1], probs[2]
        scores.append(float(pos - neg))
    # average chunk scores
    return float(np.mean(scores)) if scores else 0.0

def compute_embedding(text: str) -> np.ndarray:
    if not isinstance(text, str) or len(text) < 5:
        return np.zeros(384)  # Default zero vector for MiniLM-L6-v2 dim
    return emb_model.encode(text, convert_to_numpy=True)

# -------------------------
# Twitter fetching via tweepy
# -------------------------
def fetch_tweets_for_accounts(accounts: List[str], since_days: int = 7) -> List[Dict[str, Any]]:
    bearer_token = os.getenv('TWITTER_BEARER_TOKEN')
    if not bearer_token:
        logging.warning("TWITTER_BEARER_TOKEN not set. Skipping Twitter collection.")
        return []
    client = tweepy.Client(bearer_token=bearer_token)
    rows = []
    start_time = (datetime.utcnow() - timedelta(days=since_days)).isoformat() + 'Z'
    for acct in accounts:
        try:
            user = client.get_user(username=acct)
            tweets = client.get_users_tweets(id=user.data.id, start_time=start_time, tweet_fields=['created_at', 'text'])
            if tweets.data:
                for t in tweets.data:
                    rows.append({
                        "title": None,
                        "summary": t.text,
                        "link": f"https://twitter.com/{acct}/status/{t.id}",
                        "published": t.created_at.isoformat(),
                        "source": "Twitter",
                        "author": acct
                    })
        except Exception as e:
            logging.warning(f"Tweepy failed for {acct}: {e}")
    return rows

# -------------------------
# Main ingestion logic
# -------------------------
def ingest_all(rss_feeds=RSS_FEEDS, scrape_pages=SCRAPE_PAGES, twitter_accounts=TWITTER_ACCOUNTS, since_twitter_days=7):
    records = []

    # 1) RSS ingestion
    print("Collecting RSS feeds...")
    for src, url in rss_feeds.items():
        try:
            entries = fetch_rss_entries(url)
            for e in entries:
                link = e["link"]
                text = fetch_article_text(link)
                if not text:
                    text = e["summary"]
                published = parse_datetime(e["published"] or datetime.utcnow().isoformat())
                records.append({
                    "id": f"rss::{src}::{hash(link)}",
                    "datetime_utc": published.isoformat(),
                    "source": src,
                    "url": link,
                    "title": e["title"],
                    "content": text,
                    "raw_summary": e["summary"]
                })
        except Exception as exc:
            logging.warning(f"RSS fetch failed for {src}: {exc}")

    # 2) Scrape official pages (FOMC / ECB). We implement simple page traversal: find links, fetch article pages.
    print("Scraping official pages (basic)...")
    for src, url in scrape_pages.items():
        try:
            r = requests.get(url, timeout=15, headers={"User-Agent": "macro-collector/1.0"})
            r.raise_for_status()
            soup = BeautifulSoup(r.content, "html.parser")
            # Heuristic: find <a> links that point to press releases / statements
            links = set()
            for a in soup.find_all("a", href=True):
                href = a["href"]
                # normalize relative links
                if href.startswith("/"):
                    base = re.match(r"https?://[^/]+", url).group(0)
                    href = base + href
                if "press" in href or "statement" in href or "fomc" in href or "minutes" in href:
                    links.add(href)
            # limit to recent N links
            for link in list(links)[:40]:
                time.sleep(1)  # Polite delay
                try:
                    doc = requests.get(link, timeout=10, headers={"User-Agent": "macro-collector/1.0"})
                    doc.raise_for_status()
                    soup2 = BeautifulSoup(doc.content, "html.parser")
                    # Extract published date
                    date_tag = soup2.find("time") or soup2.find("meta", {"name":"DC.date.issued"}) or soup2.find("meta", {"property":"article:published_time"})
                    published_str = date_tag.get("datetime") if date_tag and date_tag.get("datetime") else None
                    published = parse_datetime(published_str or datetime.utcnow().isoformat())
                    # Skip if older than 7 days
                    if (datetime.utcnow() - published.replace(tzinfo=None)).days > 7:
                        continue
                    # Extract text
                    for s in soup2(["script", "style"]):
                        s.decompose()
                    paragraphs = [p.get_text(separator=" ", strip=True) for p in soup2.find_all("p")]
                    text = "\n\n".join([p for p in paragraphs if len(p) > 20])
                    records.append({
                        "id": f"scrape::{src}::{hash(link)}",
                        "datetime_utc": published.isoformat(),
                        "source": src,
                        "url": link,
                        "title": a.text.strip() or None,
                        "content": text
                    })
                except Exception as exc:
                    logging.warning(f"Failed to fetch {link}: {exc}")
        except Exception as exc:
            logging.warning(f"Scrape failed for {src}: {exc}")

    # 3) Twitter fetching via tweepy
    print("Collecting tweets (tweepy)...")
    try:
        tweets = fetch_tweets_for_accounts(twitter_accounts, since_days=since_twitter_days)
        for t in tweets:
            published = parse_datetime(t.get("published"))
            records.append({
                "id": f"twitter::{hash(t.get('link'))}",
                "datetime_utc": published.isoformat(),
                "source": "Twitter",
                "url": t.get("link"),
                "title": None,
                "content": t.get("summary"),
                "author": t.get("author")
            })
    except Exception as exc:
        logging.warning(f"Twitter collection failed: {exc}")

    # 4) Post-processing: language detection, translation, cleaning, FinBERT scoring, embeddings
    print("Post-processing items: language detection, scoring, embeddings...")
    translator = Translator()
    processed = []
    for rec in tqdm(records):
        content = rec.get("content") or ""
        # fallback to summary if content empty
        if not content and rec.get("raw_summary"):
            content = rec.get("raw_summary")
        if not content or len(content) < 10:
            logging.info(f"Skipped item {rec['id']}: content too short")
            continue
        # basic cleanup
        content_clean = re.sub(r'\s+', ' ', content).strip()
        # language detection
        try:
            lang = detect(content_clean[:1000])
        except Exception:
            lang = "unknown"
        # Auto-translate non-en texts
        translated = content_clean
        if lang != 'en' and lang != "unknown":
            try:
                translated = translator.translate(content_clean, dest='en').text
            except Exception as e:
                logging.warning(f"Translation failed for {rec['id']}: {e}")
        # Score with FinBERT
        try:
            tone = finbert_tone_score(translated)
        except Exception as e:
            logging.warning(f"FinBERT scoring failed for {rec['id']}: {e}")
            tone = 0.0
        # Embedding
        try:
            emb = compute_embedding(translated)
            emb_list = emb.tolist()
        except Exception as e:
            logging.warning(f"Embedding failed for {rec['id']}: {e}")
            emb_list = [0.0] * 384
        # Datetime normalization
        dt_parsed = parse_datetime(rec.get("datetime_utc"))
        processed.append({
            "id": rec.get("id"),
            "datetime_utc": dt_parsed.isoformat(),
            "date": dt_parsed.date().isoformat(),
            "time": dt_parsed.time().isoformat(),
            "source": rec.get("source", "other"),
            "url": rec.get("url"),
            "title": rec.get("title"),
            "content": content_clean,
            "language": lang,
            "translated_content_en": translated,
            "finbert_tone": tone,
            "embedding": emb_list,
            "author": rec.get("author", None)
        })

    # Convert to DataFrame
    df = pd.DataFrame(processed)
    if df.empty:
        print("No items collected.")
        return None

    # Dedup with existing if exists
    if os.path.exists(OUTPUT_RAW):
        df_existing = pd.read_parquet(OUTPUT_RAW)
        df = pd.concat([df_existing, df]).drop_duplicates(subset=['id'], keep='last')

    # Save raw
    df.to_parquet(OUTPUT_RAW, index=False)
    print(f"Saved raw collected items to {OUTPUT_RAW} ({len(df)} rows)")
    return df

# -------------------------
# Daily aggregation
# -------------------------
def aggregate_daily(df: pd.DataFrame, credibility_map: Dict[str, float] = SOURCE_CREDIBILITY):
    """
    Produce daily aggregated macro score table with:
     - date, n_items, avg_finbert_tone_weighted, median_tone, std_tone
     - optionally include top-k topics / top sources
    """
    # ensure datetime
    df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
    df['date'] = df['datetime_utc'].dt.date
    # credibility weight
    df['cred'] = df['source'].map(credibility_map).fillna(credibility_map.get('other', 0.5))
    # recency weight (intra-day prioritization): weight by hour (newer = slightly bigger). simple: 1 + fraction of day
    df['hour_frac'] = df['datetime_utc'].dt.hour / 24.0
    df['weight'] = df['cred'] * (1 + df['hour_frac'])
    # Weighted average finbert tone per day
    def weighted_avg(group):
        weights = group['weight'].values
        vals = group['finbert_tone'].values
        if weights.sum() == 0:
            return 0.0
        return float(np.dot(weights, vals) / weights.sum())

    daily = df.groupby('date').apply(lambda g: pd.Series({
        "date": g.name,
        "n_items": len(g),
        "finbert_tone_wavg": weighted_avg(g),
        "finbert_tone_med": float(np.median(g['finbert_tone'])),
        "finbert_tone_std": float(np.std(g['finbert_tone'])),
        "top_source": g['source'].value_counts().idxmax()
    })).reset_index(drop=True)
    # Dedup with existing if exists
    daily['date'] = pd.to_datetime(daily['date'])
    if os.path.exists(OUTPUT_DAILY):
        daily_existing = pd.read_parquet(OUTPUT_DAILY)
        daily = pd.concat([daily_existing, daily]).drop_duplicates(subset=['date'], keep='last')
    # Save
    daily.to_parquet(OUTPUT_DAILY, index=False)
    print(f"Saved daily aggregated signals to {OUTPUT_DAILY} ({len(daily)} days)")
    return daily

# -------------------------
# Entrypoint
# -------------------------
if __name__ == "__main__":
    print("Starting macro data collection pipeline...")
    df_raw = ingest_all()
    if df_raw is not None and not df_raw.empty:
        df_daily = aggregate_daily(df_raw)
        print("Done.")
    else:
        print("Nothing to aggregate. Exiting.")

#NG----------------------------------------------------------------------------------------------------------------------

Loading NLP models... (this may take a while on first run)
Starting macro data collection pipeline...
Scraping official pages (basic)...


C:\Users\Admin\AppData\Local\Temp\ipykernel_28932\1154092775.py:285: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  published = parse_datetime(published_str or datetime.utcnow().isoformat())
C:\Users\Admin\AppData\Local\Temp\ipykernel_28932\1154092775.py:287: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  if (datetime.utcnow() - published.replace(tzinfo=None)).days > 7:
C:\Users\Admin\anaconda3\Lib\html\parser.py:189: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable.

Post-processing items: language detection, scoring, embeddings...


100%|█████████████████████████████████████████████| 74/74 [02:27<00:00,  2.00s/it]

Saved raw collected items to ./data/raw/macro_raw.parquet (73 rows)
Saved daily aggregated signals to ./data/daily/macro_daily_signals.parquet (1 days)
Done.



C:\Users\Admin\AppData\Local\Temp\ipykernel_28932\1154092775.py:423: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily = df.groupby('date').apply(lambda g: pd.Series({


In [5]:
import os, tweepy
os.environ["TWITTER_BEARER_TOKEN"] = "AAAAAAAAAAAAAAAAAAAAAF%2B%2B4QEAAAAAHxzpSLBfZGyXSGlRrq2I46Ti%2FkY%3DgmwGjYbEwqnWby3CWaXQbG84oNp2VYLeedSiFabLzkkd1FaQvB"
client = tweepy.Client(bearer_token=os.getenv("TWITTER_BEARER_TOKEN"))
print(client)

In [6]:
"""
collect_macro_data.py

Collecte macro "end-to-end":
 - RSS feeds (ECB, IMF, Reuters...)
 - Scraping officiels (FOMC statements/minutes, ECB press releases) - now with historical loop for 2018-2025
 - Twitter/X scraping via tweepy (comptes officiels) - requires TWITTER_BEARER_TOKEN env var, adjusted for historical
 - Language detection, optional translation using googletrans
 - FinBERT tone scoring (hawkish/dovish proxy)
 - Sentence-transformers embeddings
 - Store: raw parquet and daily aggregated parquet signals

Sorties:
 - /data/raw/macro_raw.parquet
 - /data/daily/macro_daily_signals.parquet
"""

import os
import json
from datetime import datetime, timezone, timedelta
import time
import re
import logging
from typing import List, Dict, Any

import feedparser
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from langdetect import detect, DetectorFactory
from newspaper import Article
from tqdm import tqdm
import dateutil.parser

# NLP
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

# Twitter
import tweepy

# Translation
from googletrans import Translator

# Ensure deterministic langdetect
DetectorFactory.seed = 0

# -------------------------
# Configuration
# -------------------------
OUTPUT_RAW = "data/macro_raw.parquet"
OUTPUT_DAILY = "data/macro_daily_signals.parquet"
os.makedirs(os.path.dirname(OUTPUT_RAW), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_DAILY), exist_ok=True)

START_YEAR = 2018
END_YEAR = 2025  # Inclusive

# RSS & scraping sources (extend as needed)
RSS_FEEDS = {
    "ECB_press": "https://www.ecb.europa.eu/rss/press.xml",
    "IMF_news": "https://www.imf.org/external/rss/bottom.xml",  # Recent; historical via scraping if needed
}

# Official pages to scrape (base URLs; year-specific in loop)
SCRAPE_PAGES = {
    "FOMC_press": "https://www.federalreserve.gov/newsevents/pressreleases/{year}-press.htm",
    "ECB_press": "https://www.ecb.europa.eu/press/pr/date/{year}/html/index.en.html",
    "FOMC_minutes": "https://www.federalreserve.gov/monetarypolicy/fomc_historical_year.htm",  # Needs special handling
}

# Twitter accounts
TWITTER_ACCOUNTS = [
    "federalreserve", "ecb", "bankofengland", "boj_official_en"
]

# Model names
FINBERT_MODEL = "yiyanghkust/finbert-tone"
SENTENCE_EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Credibility scores
SOURCE_CREDIBILITY = {
    "ECB_press": 1.0,
    "FOMC_press": 1.0,
    "IMF_news": 0.9,
    "Twitter": 0.6,
    "other": 0.5
}

MAX_TOKENS = 500

# -------------------------
# Helpers
# -------------------------
def fetch_rss_entries(feed_url: str) -> List[Dict[str, Any]]:
    feed = feedparser.parse(feed_url)
    entries = []
    for e in feed.entries:
        published = e.get("published") or e.get("updated") or None
        entries.append({
            "title": e.get("title", ""),
            "summary": e.get("summary", "") or e.get("description", ""),
            "link": e.get("link", ""),
            "published": published
        })
    return entries

def fetch_article_text(url: str) -> str:
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        if text and len(text) > 50:
            return text
    except Exception:
        pass
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "macro-collector/1.0"})
        r.raise_for_status()
        soup = BeautifulSoup(r.content, "html.parser")
        for s in soup(["script", "style"]):
            s.decompose()
        paragraphs = [p.get_text(separator=" ", strip=True) for p in soup.find_all("p")]
        text = "\n\n".join([p for p in paragraphs if len(p) > 20])
        return text
    except Exception:
        return ""

def parse_datetime(s, url=None):
    if s:
        try:
            dt = dateutil.parser.parse(s, fuzzy=True)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return pd.Timestamp(dt)
        except Exception:
            pass
    if url:
        date_match = re.search(r'(\d{4})(\d{2})(\d{2})', url)
        if date_match:
            date_str = f"{date_match.group(1)}-{date_match.group(2)}-{date_match.group(3)}"
            try:
                dt = pd.to_datetime(date_str, utc=True)
                return dt
            except Exception:
                pass
    return pd.Timestamp(datetime.utcnow(), tz='UTC')

# -------------------------
# NLP
# -------------------------
print("Loading NLP models...")
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    fin_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
    fin_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL).to(device)
    emb_model = SentenceTransformer(SENTENCE_EMB_MODEL, device=device)
except Exception as e:
    logging.error(f"Failed to load NLP models: {e}")
    raise

def chunk_text(text: str, max_tokens: int = MAX_TOKENS) -> List[str]:
    parts = re.split(r'\n{2,}|\.\s+', text)
    chunks = []
    cur = ""
    for p in parts:
        if len((cur + " " + p).split()) > max_tokens:
            if cur:
                chunks.append(cur.strip())
            cur = p
        else:
            cur = (cur + " " + p).strip()
    if cur:
        chunks.append(cur.strip())
    return [c for c in chunks if len(c) > 20]

def finbert_tone_score(text: str) -> float:
    if not isinstance(text, str) or len(text) < 10:
        return 0.0
    chunks = chunk_text(text)
    scores = []
    for ch in chunks:
        inputs = fin_tokenizer(ch, truncation=True, padding=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = fin_model(**inputs).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy().flatten()
        neg, neu, pos = probs[0], probs[1], probs[2]
        scores.append(float(pos - neg))
    return float(np.mean(scores)) if scores else 0.0

def compute_embedding(text: str) -> np.ndarray:
    if not isinstance(text, str) or len(text) < 5:
        return np.zeros(384)
    return emb_model.encode(text, convert_to_numpy=True)

# -------------------------
# Twitter
# -------------------------
def fetch_tweets_for_accounts(accounts: List[str], since_days: int = (datetime.utcnow() - datetime(2018, 1, 1)).days) -> List[Dict[str, Any]]:
    bearer_token = os.getenv('TWITTER_BEARER_TOKEN')
    if not bearer_token:
        logging.warning("TWITTER_BEARER_TOKEN not set. Skipping Twitter collection.")
        return []
    client = tweepy.Client(bearer_token=bearer_token)
    rows = []
    start_time = (datetime.utcnow() - timedelta(days=since_days)).isoformat() + 'Z'
    end_time = datetime(2025, 1, 1).isoformat() + 'Z'
    for acct in accounts:
        try:
            user = client.get_user(username=acct)
            tweets = client.get_users_tweets(id=user.data.id, start_time=start_time, end_time=end_time, tweet_fields=['created_at', 'text'], max_results=100)
            if tweets.data:
                for t in tweets.data:
                    rows.append({
                        "title": None,
                        "summary": t.text,
                        "link": f"https://twitter.com/{acct}/status/{t.id}",
                        "published": t.created_at.isoformat(),
                        "source": "Twitter",
                        "author": acct
                    })
            # Note: For full historical, pagination is needed; this gets recent 100. For more, use loop with next_token
        except Exception as e:
            logging.warning(f"Tweepy failed for {acct}: {e}")
    return rows

# -------------------------
# Main ingestion
# -------------------------
def ingest_all(rss_feeds=RSS_FEEDS, scrape_pages=SCRAPE_PAGES, twitter_accounts=TWITTER_ACCOUNTS, since_twitter_days=(datetime.utcnow() - datetime(2018, 1, 1)).days):
    records = []

    # 1) RSS - recent only
    print("Collecting RSS feeds (recent only)...")
    for src, url in rss_feeds.items():
        try:
            entries = fetch_rss_entries(url)
            for e in entries:
                link = e["link"]
                text = fetch_article_text(link)
                if not text:
                    text = e["summary"]
                published = parse_datetime(e["published"], url=link)
                if published.year < START_YEAR or published.year > END_YEAR:
                    continue
                records.append({
                    "id": f"rss::{src}::{hash(link)}",
                    "datetime_utc": published.isoformat(),
                    "source": src,
                    "url": link,
                    "title": e["title"],
                    "content": text,
                    "raw_summary": e["summary"]
                })
        except Exception as exc:
            logging.warning(f"RSS fetch failed for {src}: {exc}")

    # 2) Scrape - historical by year
    print("Scraping official pages (historical 2018-2025)...")
    for src, base_url in scrape_pages.items():
        for year in range(START_YEAR, END_YEAR + 1):
            url = base_url.format(year=year)
            try:
                r = requests.get(url, timeout=15, headers={"User-Agent": "macro-collector/1.0"})
                r.raise_for_status()
                soup = BeautifulSoup(r.content, "html.parser")
                links = set()
                for a in soup.find_all("a", href=True):
                    href = a["href"]
                    if href.startswith("/"):
                        base = re.match(r"https?://[^/]+", url).group(0)
                        href = base + href
                    if "press" in href or "statement" in href or "fomc" in href or "minutes" in href or "release" in href:
                        links.add(href)
                for link in links:
                    time.sleep(1)
                    try:
                        doc = requests.get(link, timeout=10, headers={"User-Agent": "macro-collector/1.0"})
                        doc.raise_for_status()
                        soup2 = BeautifulSoup(doc.content, "html.parser")
                        date_tag = soup2.find("time") or soup2.find("meta", {"name":"DC.date.issued"}) or soup2.find("meta", {"property":"article:published_time"}) or soup2.find('p', class_='article__time')
                        published_str = date_tag.get("datetime") or date_tag.get("content") or (date_tag.text.strip() if date_tag else None)
                        published = parse_datetime(published_str, url=link)
                        if published.year != year:
                            continue  # Ensure year match
                        for s in soup2(["script", "style"]):
                            s.decompose()
                        paragraphs = [p.get_text(separator=" ", strip=True) for p in soup2.find_all("p")]
                        text = "\n\n".join([p for p in paragraphs if len(p) > 20])
                        records.append({
                            "id": f"scrape::{src}::{year}::{hash(link)}",
                            "datetime_utc": published.isoformat(),
                            "source": src,
                            "url": link,
                            "title": a.text.strip() or None,
                            "content": text
                        })
                    except Exception as exc:
                        logging.warning(f"Failed to fetch {link}: {exc}")
            except Exception as exc:
                logging.warning(f"Scrape failed for {src} year {year}: {exc}")

    # 3) Twitter - historical (note: may require premium API for full access; basic limited)
    print("Collecting tweets (historical 2018-2025)...")
    try:
        tweets = fetch_tweets_for_accounts(twitter_accounts, since_days=since_twitter_days)
        for t in tweets:
            published = parse_datetime(t.get("published"))
            if published.year < START_YEAR or published.year > END_YEAR:
                continue
            records.append({
                "id": f"twitter::{hash(t.get('link'))}",
                "datetime_utc": published.isoformat(),
                "source": "Twitter",
                "url": t.get("link"),
                "title": None,
                "content": t.get("summary"),
                "author": t.get("author")
            })
    except Exception as exc:
        logging.warning(f"Twitter collection failed: {exc}")

    # 4) Post-processing
    print("Post-processing items...")
    translator = Translator()
    processed = []
    for rec in tqdm(records):
        content = rec.get("content") or ""
        if not content and rec.get("raw_summary"):
            content = rec.get("raw_summary")
        if not content or len(content) < 10:
            logging.info(f"Skipped item {rec['id']}: content too short")
            continue
        content_clean = re.sub(r'\s+', ' ', content).strip()
        try:
            lang = detect(content_clean[:1000])
        except Exception:
            lang = "unknown"
        translated = content_clean
        if lang != 'en' and lang != "unknown":
            try:
                translated = translator.translate(content_clean, dest='en').text
            except Exception as e:
                logging.warning(f"Translation failed for {rec['id']}: {e}")
        try:
            tone = finbert_tone_score(translated)
        except Exception as e:
            logging.warning(f"FinBERT failed for {rec['id']}: {e}")
            tone = 0.0
        try:
            emb = compute_embedding(translated)
            emb_list = emb.tolist()
        except Exception as e:
            logging.warning(f"Embedding failed for {rec['id']}: {e}")
            emb_list = [0.0] * 384
        dt_parsed = parse_datetime(rec.get("datetime_utc"))
        processed.append({
            "id": rec.get("id"),
            "datetime_utc": dt_parsed.isoformat(),
            "date": dt_parsed.date().isoformat(),
            "time": dt_parsed.time().isoformat(),
            "source": rec.get("source", "other"),
            "url": rec.get("url"),
            "title": rec.get("title"),
            "content": content_clean,
            "language": lang,
            "translated_content_en": translated,
            "finbert_tone": tone,
            "embedding": emb_list,
            "author": rec.get("author", None)
        })

    df = pd.DataFrame(processed)
    if df.empty:
        print("No items collected.")
        return None

    if os.path.exists(OUTPUT_RAW):
        df_existing = pd.read_parquet(OUTPUT_RAW)
        df = pd.concat([df_existing, df]).drop_duplicates(subset=['id'], keep='last')

    df.to_parquet(OUTPUT_RAW, index=False)
    print(f"Saved raw to {OUTPUT_RAW} ({len(df)} rows)")
    return df

# -------------------------
# Daily aggregation
# -------------------------
def aggregate_daily(df: pd.DataFrame, credibility_map: Dict[str, float] = SOURCE_CREDIBILITY):
    df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
    df['date'] = df['datetime_utc'].dt.date
    df['cred'] = df['source'].map(credibility_map).fillna(credibility_map.get('other', 0.5))
    df['hour_frac'] = df['datetime_utc'].dt.hour / 24.0
    df['weight'] = df['cred'] * (1 + df['hour_frac'])
    def weighted_avg(group):
        weights = group['weight'].values
        vals = group['finbert_tone'].values
        if weights.sum() == 0:
            return 0.0
        return float(np.dot(weights, vals) / weights.sum())

    daily = df.groupby('date').apply(lambda g: pd.Series({
        "date": g.name,
        "n_items": len(g),
        "finbert_tone_wavg": weighted_avg(g),
        "finbert_tone_med": float(np.median(g['finbert_tone'])),
        "finbert_tone_std": float(np.std(g['finbert_tone'])),
        "top_source": g['source'].value_counts().idxmax()
    })).reset_index(drop=True)
    daily['date'] = pd.to_datetime(daily['date'])
    if os.path.exists(OUTPUT_DAILY):
        daily_existing = pd.read_parquet(OUTPUT_DAILY)
        daily = pd.concat([daily_existing, daily]).drop_duplicates(subset=['date'], keep='last')
    daily.to_parquet(OUTPUT_DAILY, index=False)
    print(f"Saved daily to {OUTPUT_DAILY} ({len(daily)} days)")
    return daily

# -------------------------
# Entrypoint
# -------------------------
if __name__ == "__main__":
    print("Starting historical macro data collection (2018-2025)...")
    df_raw = ingest_all()
    if df_raw is not None and not df_raw.empty:
        #df_daily = aggregate_daily(df_raw)
        print("Done.")
    else:
        print("Nothing to aggregate. Exiting.")

Loading NLP models...


C:\Users\Admin\AppData\Local\Temp\ipykernel_4716\3896320488.py:202: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def fetch_tweets_for_accounts(accounts: List[str], since_days: int = (datetime.utcnow() - datetime(2018, 1, 1)).days) -> List[Dict[str, Any]]:
C:\Users\Admin\AppData\Local\Temp\ipykernel_4716\3896320488.py:233: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def ingest_all(rss_feeds=RSS_FEEDS, scrape_pages=SCRAPE_PAGES, twitter_accounts=TWITTER_ACCOUNTS, since_twitter_days=(datetime.utcnow() - datetime(2018, 1, 1)).days):


Starting historical macro data collection (2018-2025)...


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Scraping official pages (historical 2018-2025)...


C:\Users\Admin\AppData\Local\Temp\ipykernel_4716\3896320488.py:209: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = (datetime.utcnow() - timedelta(days=since_days)).isoformat() + 'Z'


The `start_time` query parameter value [2018-01-01T20:25:13.298475Z] is not a valid RFC3339 date-time, must follow pattern 'yyyy-MM-dd'T'HH:mm:ss[.SSS]X.
Too Many Requests
Too Many Requests
Too Many Requests


Post-processing items...


C:\Users\Admin\AppData\Local\Temp\ipykernel_4716\3896320488.py:347: RuntimeWarning: coroutine 'Translator.translate' was never awaited
  logging.warning(f"Translation failed for {rec['id']}: {e}")
100%|██████████| 1294/1294 [20:43<00:00,  1.04it/s]


Saved raw to data/macro_raw.parquet (1293 rows)
Done.


In [18]:
my_data = pd.read_parquet('data/macro_raw.parquet')
print(my_data.shape)
print(my_data['date'].value_counts())
my_data.head()

(1293, 13)
date
2018-12-20    6
2022-05-23    6
2025-09-17    5
2019-11-19    5
2019-03-06    4
             ..
2025-07-10    1
2025-06-13    1
2025-09-04    1
2025-08-22    1
2025-01-08    1
Name: count, Length: 898, dtype: int64


,id,datetime_utc,date,time,source,url,title,content,language,translated_content_en,finbert_tone,embedding,author
0,rss::ECB_press::5902712044358368242,2025-09-26T15:00:00+02:00,2025-09-26,15:00:00,ECB_press,https://www.ecb.europa.eu//press/govcdec/other...,Decisions taken by the Governing Council of th...,Decisions taken by the Governing Council of th...,en,Decisions taken by the Governing Council of th...,-0.999751,"[-0.005181720945984125, -0.000976837589405477,...",None
1,rss::ECB_press::-3831607764618867839,2025-09-26T10:00:00+02:00,2025-09-26,10:00:00,ECB_press,https://www.ecb.europa.eu//press/pr/date/2025/...,ECB Consumer Expectations Survey results â€“ A...,Compared with July 2025: median consumer perce...,en,Compared with July 2025: median consumer perce...,-0.921554,"[0.014337118715047836, -0.04651716351509094, -...",None
2,rss::ECB_press::-1620183602607990191,2025-09-26T09:15:00+02:00,2025-09-26,09:15:00,ECB_press,https://www.ecb.europa.eu//press/key/date/2025...,Piero Cipollone: Preparing the future of payme...,"Keynote speech by Piero Cipollone, Member of t...",en,"Keynote speech by Piero Cipollone, Member of t...",-0.477997,"[-0.057935696095228195, 0.020412525162100792, ...",None
3,rss::ECB_press::725466049031953345,2025-09-26T09:15:00+02:00,2025-09-26,09:15:00,ECB_press,https://www.ecb.europa.eu//press/pr/date/2025/...,ECB presents findings from digital euro innova...,Experimentation with almost 70 market particip...,en,Experimentation with almost 70 market particip...,-0.099782,"[-0.0196444820612669, -0.015468696132302284, -...",None
4,rss::ECB_press::1278098667427235584,2025-09-25T18:30:00+02:00,2025-09-25,18:30:00,ECB_press,https://www.ecb.europa.eu//press/inter/date/20...,Piero Cipollone: Interview with Milano Finanza,"Interview with Piero Cipollone, Member of the ...",en,"Interview with Piero Cipollone, Member of the ...",-0.767465,"[-0.05637403577566147, -0.007352667395025492, ...",None
